## **TPKAGGLE**



In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# URLs de tu repositorio de GitHub (formato Raw)
url_test = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_test.csv'
url_comportamiento = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/comportamiento_bancario.csv'
url_train = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_train.csv'

# Cargar los datos directamente desde GitHub
df_solicitudes_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comportamiento)
df_solicitudes = pd.read_csv(url_train)

In [6]:

df_comportamiento['Mes_Referencia'] = pd.to_datetime(df_comportamiento['Mes_Referencia'])
df_solicitudes['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes['Fecha_Solicitud'])
df_solicitudes_test['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes_test['Fecha_Solicitud'])

def agregar_comportamiento(df_solicitudes_base, df_comportamiento):
    df_temp = pd.merge(
        left=df_solicitudes_base[['ID_Solicitud','ID_Usuario','Fecha_Solicitud']],
        right=df_comportamiento[['ID_Usuario','Mes_Referencia','Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']],
        on='ID_Usuario',
        how='left'
    )
    # Solo meses anteriores a CADA fecha de solicitud puntual
    df_temp = df_temp[df_temp['Mes_Referencia'] < df_temp['Fecha_Solicitud']]
    return df_temp.groupby('ID_Solicitud')[['Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']].mean()

df_comportamiento_agrupado_train = agregar_comportamiento(df_solicitudes, df_comportamiento)
df_cruzado = pd.merge(df_solicitudes, df_comportamiento_agrupado_train, on='ID_Solicitud', how='left')
df_cruzado.sort_values(by='ID_Usuario', ascending=True)

# Para test, con sus PROPIAS fechas (2026-01):
df_comportamiento_agrupado_test = agregar_comportamiento(df_solicitudes_test, df_comportamiento)
df_solicitudes_test_cruzado = pd.merge(
    left=df_solicitudes_test,
    right=df_comportamiento_agrupado_test,
    on='ID_Solicitud',
    how='left'
  )

df_solicitudes_test_cruzado.isnull().sum(),df_cruzado.isnull().sum()

(ID_Solicitud                0
 ID_Usuario                  0
 Monto_Solicitado            0
 Tasa_Interes                0
 Ingreso_Mensual             0
 Motivo_Prestamo             0
 Fecha_Solicitud             0
 Saldo_Promedio_Mensual      0
 Dias_Atraso_Otras_Deudas    0
 dtype: int64,
 ID_Solicitud                0
 ID_Usuario                  0
 Monto_Solicitado            0
 Tasa_Interes                0
 Ingreso_Mensual             0
 Motivo_Prestamo             0
 Fecha_Solicitud             0
 Estado_Final                0
 Saldo_Promedio_Mensual      0
 Dias_Atraso_Otras_Deudas    0
 dtype: int64)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn import metrics

#Creamos el mapper
mapper = DataFrameMapper([
(['Monto_Solicitado'],[ StandardScaler()]),
(['Saldo_Promedio_Mensual'],[StandardScaler(),IterativeImputer()]),
(['Dias_Atraso_Otras_Deudas'],[StandardScaler(),IterativeImputer()])
])
#Creamos el pipeline
pipeline = Pipeline(
    [('mapper', mapper),
     ('classifier', LogisticRegression(random_state=70))
])


#Entrenamos con train
pipeline.fit(df_cruzado, df_cruzado['Estado_Final'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Pipeline(steps=[('mapper',
                 DataFrameMapper(drop_cols=[],
                                 features=[(['Monto_Solicitado'],
                                            [StandardScaler()]),
                                           (['Saldo_Promedio_Mensual'],
                                            [StandardScaler(),
                                             IterativeImputer()]),
                                           (['Dias_Atraso_Otras_Deudas'],
                                            [StandardScaler(),
                                             IterativeImputer()])])),
                ('classifier', LogisticRegression(random_state=70))])

In [8]:
#METRICAS PARA VER PESOS DE VARIABLES
clf = pipeline.named_steps['classifier']
clf.coef_, pipeline.named_steps['mapper'].transformed_names_


(array([[-0.55411505,  0.93489442, -0.86882441]]),
 ['Monto_Solicitado', 'Saldo_Promedio_Mensual', 'Dias_Atraso_Otras_Deudas'])

In [9]:
new_predictions_prob = pipeline.predict_proba(df_solicitudes_test_cruzado)[:, 1]
# AGREGAMOS A LAS SOLICITUDES LA COLUMNA DE PROBABILIDAD DE QUE SEA PAGADO EL PRESTAMO
df_solicitudes_test_cruzado['Prob_Pago'] = new_predictions_prob
df_solicitudes_test_cruzado.sample(5)

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas,Prob_Pago
653,33070,26914,900,0.2935,2442,Consumo,2026-01-01,1972.268636,4.772727,0.936936
1538,33955,26696,5100,0.3219,5305,Negocio,2026-01-01,8425.335909,10.000000,0.958244
1996,34413,26386,6500,0.2930,5525,Vivienda,2026-01-01,11410.571818,5.181818,0.986437
724,33141,27599,1800,0.2881,2758,Consumo,2026-01-01,5395.309474,7.210526,0.953856
1555,33972,27776,500,0.1808,2183,Educacion,2026-01-01,1657.957391,10.086957,0.877669


In [10]:
#MECANISMO PARA NO PRESTAR DE MáS
monto_solicitado_total = 0
monto_a_ganar=0
#AñADIMOS UNA COLUMNA A LAS SOLICITUDES CON ESTADO = 0
df_solicitudes_test_cruzado['Estado'] = 0
#CREAMOS LISTA DE INDICES DE LAS FILAS DE LAS SOLICITUDES QUE APROBAREMOS
approved_indices = []

#ITERAMOS CADA FILA
for index, row in df_solicitudes_test_cruzado.sort_values(by='Prob_Pago', ascending=False).iterrows():
    # SI ES POSITIVA Y TENEMOS EL DINERO PARA PRESTAR, PRESTAMOS, GUARDAMOS INDICE Y ACTUALIZAMOS TOTAL PRESTADO
    if ((monto_solicitado_total + row['Monto_Solicitado']) < 2000000) and (row['Prob_Pago']>0.80):
        monto_solicitado_total += row['Monto_Solicitado']
        monto_a_ganar+= row['Tasa_Interes']*row['Monto_Solicitado']
        approved_indices.append(index)

# ACTUALIZAMOS EL ESTADO A 1, DE DICHAS SOLICITUDES QUE APROBAMOS
df_solicitudes_test_cruzado.loc[approved_indices, 'Estado'] = 1
monto_solicitado_total,monto_a_ganar


(1999900, 539882.8899999999)

In [11]:
df_resultados = df_solicitudes_test_cruzado[['ID_Solicitud', 'Estado']].copy()
df_resultados.rename(columns={'Estado': 'Decision_Aprobacion'}, inplace=True)

# Exportar a CSV
df_resultados.to_csv('decisiones_aprobacion.csv', index=False)

print('Archivo decisiones_aprobacion.csv creado exitosamente.')
display(df_resultados.head())

Archivo decisiones_aprobacion.csv creado exitosamente.


,ID_Solicitud,Decision_Aprobacion
0,32417,0
1,32418,1
2,32419,0
3,32420,0
4,32421,0
